In [12]:
from pathlib import Path
import os
from dotenv import load_dotenv
from operator import itemgetter

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough


In [ ]:
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'ingestion.py').exists():
    PROJECT_DIR = Path('/Users/abtinzandi/Documents/projects/LLM-GANs-DP/LLM-GANs-Projects/LangChain/udemy_course/Medium_analyzer')

load_dotenv(PROJECT_DIR / '.env')
print('Loaded .env from:', PROJECT_DIR / '.env')
print('INDEX_NAME:', os.getenv('INDEX_NAME'))

Loaded .env from: /Users/abtinzandi/Documents/projects/LLM-GANs-DP/LLM-GANs-Projects/LangChain/udemy_course/Medium_analyzer/.env
INDEX_NAME: medium-analyer


In [15]:
QUERY = "what is Pinecone in machine learning?"

In [ ]:
embeddings = OpenAIEmbeddings()
llm = ChatOpenAI()

In [3]:
openai_api_key = os.getenv('OPENAI_API_KEY')
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found in environment variables.")

index_name = os.getenv('INDEX_NAME')
if not index_name:
    raise ValueError("INDEX_NAME not found in environment variables.")

In [ ]:


embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)

vectorstore = PineconeVectorStore(
    index_name=index_name,
    embedding=embeddings
)

print('Connected to Pinecone index:', index_name)

/Users/abtinzandi/Documents/projects/LLM-GANs-DP/LLM-GANs-Projects/LangChain/udemy_course/Medium_analyzer/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Connected to Pinecone index: medium-analyer


In [6]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print('Retriever created with k=3')

Retriever created with k=3


In [ ]:


retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

prompt_template = ChatPromptTemplate.from_template(
   template= """Answer the question based only on the following context:

{context}

Question: {question}

Provide a detailed answer:"""
)

prompt_template

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n\n{context}\n\nQuestion: {question}\n\nProvide a detailed answer:'), additional_kwargs={})])

In [8]:
def format_docs(docs):
    """Format retrieved documents into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)

In [14]:
def create_retrieval_chain_with_lcel():

    chain = (
        RunnablePassthrough.assign(
            context=itemgetter("question") | retriever | format_docs
        ) 
        | prompt_template
        | llm
        | StrOutputParser()
    )

    return chain

In [17]:

print("\n" + "=" * 70)
print("IMPLEMENTATION 2: With LCEL - Better Approach")
print("=" * 70)
print("Why LCEL is better:")
print("- More concise and declarative")
print("- Built-in streaming: chain.stream()")
print("- Built-in async: chain.ainvoke()")
print("- Easy to compose with other chains")
print("- Better for production use")
print("=" * 70)

chain_with_lcel = create_retrieval_chain_with_lcel()
result_with_lcel = chain_with_lcel.invoke({"question": QUERY})
print("\nAnswer:")
print(result_with_lcel)


IMPLEMENTATION 2: With LCEL - Better Approach
Why LCEL is better:
- More concise and declarative
- Built-in streaming: chain.stream()
- Built-in async: chain.ainvoke()
- Easy to compose with other chains
- Better for production use

Answer:
Pinecone is a platform designed specifically for efficient retrieval of similar data points based on their vector representations in the field of machine learning. It is optimized for speed and scalability, allowing it to handle large-scale ML applications with millions or billions of data points. Pinecone offers infrastructure management and maintenance to its users, making it easier to manage and operate ML workflows.

In addition, Pinecone is known for its ability to handle high query throughput and low latency search, ensuring quick and efficient retrieval of data points. The platform is also highly secure, meeting the security needs of businesses and organizations working with sensitive data.

One of the key features of Pinecone is its user-fr